# Hyperlink-Aware Resume RAG with Semantic Cache

This notebook implements the complete architecture for a free-tier friendly, locally executing Resume RAG system. It features PyMuPDF-based hyperlink extraction, a dual-layer semantic query cache, and FAISS-based vector retrieval.

### Step 1: Install Dependencies
All libraries used here are free and open-source. They do not require any paid APIs.

In [2]:
!pip -q install pymupdf sentence-transformers faiss-cpu transformers accelerate torch pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 97.6 MB/s eta 0:00:00


### Step 2: Imports & Global Configuration

In [3]:
import os
import re
import json
import hashlib
import pickle
from typing import List, Dict, Any, Optional

import fitz  # PyMuPDF
import numpy as np
import faiss
import torch
from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Configure device (uses GPU if available in Colab, falls back cleanly to CPU)
DEVICE = 0 if torch.cuda.is_available() else -1
print(f"Executing on: {'GPU (CUDA)' if DEVICE == 0 else 'CPU'}")

# Free local Hugging Face models
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
QA_MODEL_NAME = "deepset/roberta-base-squad2"  # Extractive QA model optimized for CPU/GPU

SEMANTIC_CACHE_THRESHOLD = 0.88
TOP_K = 5

CACHE_PATH = "semantic_query_cache.json"
INDEX_PATH = "resume_faiss.index"
CHUNKS_PATH = "resume_chunks.pkl"

Executing on: GPU (CUDA)


### Step 3: Upload Resume PDF

In [4]:
# Upload your resume PDF file
uploaded = files.upload()
pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]

if not pdf_files:
    raise ValueError("Please upload a valid PDF resume file.")

PDF_PATH = pdf_files[0]
print(f"Successfully loaded: {PDF_PATH}")

Saving Syamantak_Mukherjee_ML_Resume.pdf to Syamantak_Mukherjee_ML_Resume.pdf
Successfully loaded: Syamantak_Mukherjee_ML_Resume.pdf


### Step 4: Hyperlink-Aware Extraction & URL Classification
Extracts bounding boxes and links hidden behind LaTeX formatting.

In [5]:
def sha256_file(path: str) -> str:
    """Computes a unique hash for the document to scope cache entries."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def clean_url(url: str) -> str:
    if not url:
        return ""
    url = url.strip()
    return url[7:] if url.lower().startswith("mailto:") else url

def classify_resume_url(url: str) -> str:
    """Categorizes embedded URLs into searchable resume domain types."""
    u = (url or "").lower().strip()
    if u.startswith("mailto:") or re.match(r"^[\w\.-]+@[\w\.-]+\.\w+$", u):
        return "email"
    if "linkedin.com" in u:
        return "linkedin"
    if "github.com" in u or "gitlab.com" in u or "bitbucket.org" in u:
        return "github"
    if any(domain in u for domain in ["credly.com", "coursera.org", "udemy.com", "edx.org"]):
        return "certification"
    if any(domain in u for domain in ["leetcode.com", "hackerrank.com"]):
        return "coding_profile"
    if any(domain in u for domain in ["vercel.app", "netlify.app", "herokuapp.com", "render.com", "demo"]):
        return "project_demo"
    return "portfolio" if "portfolio" in u else "other"

def rects_overlap(rect1, rect2, tolerance=3) -> bool:
    """Checks if a text bounding box overlaps with a PDF link rectangle."""
    r1, r2 = fitz.Rect(rect1), fitz.Rect(rect2)
    r1.x0 -= tolerance; r1.y0 -= tolerance
    r1.x1 += tolerance; r1.y1 += tolerance
    return r1.intersects(r2)

def extract_pdf_with_links(pdf_path: str) -> Dict[str, Any]:
    doc = fitz.open(pdf_path)
    document_hash = sha256_file(pdf_path)
    pages, global_links = [], {}

    for page_num, page in enumerate(doc, start=1):
        text_dict = page.get_text("dict")
        spans = []
        for block in text_dict.get("blocks", []):
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    if text := span.get("text", "").strip():
                        spans.append({"text": text, "bbox": list(span.get("bbox")), "page": page_num})

        page_links = []
        for link in page.get_links():
            uri, link_rect = link.get("uri"), link.get("from")
            if not uri or not link_rect:
                continue

            # Map hidden URL to visible text using bounding box spatial intersection
            matched_texts = [item["text"] for item in spans if rects_overlap(item["bbox"], link_rect)]
            label = " ".join(matched_texts).strip() or None
            link_type = classify_resume_url(uri)
            norm_url = clean_url(uri)

            page_links.append({"label": label or link_type, "url": norm_url, "type": link_type, "page": page_num})
            global_links.setdefault(link_type, [])
            if norm_url not in global_links[link_type]:
                global_links[link_type].append(norm_url)

        pages.append({"page": page_num, "text": clean_text(page.get_text("text")), "links": page_links})

    return {"document_hash": document_hash, "pages": pages, "global_links": global_links}

structured_data = extract_pdf_with_links(PDF_PATH)
print(f"Extraction Complete! Document Hash: {structured_data['document_hash'][:12]}...")
print("Detected Hyperlinks by Category:", json.dumps(structured_data["global_links"], indent=2))

Extraction Complete! Document Hash: 40faee11ae74...
Detected Hyperlinks by Category: {
  "linkedin": [
    "https://www.linkedin.com/in/syamantak-mukherjee/"
  ],
  "github": [
    "https://github.com/mukherjee-syamantak",
    "https://github.com/mukherjee-syamantak/LLM-Operations_Workflow",
    "https://github.com/mukherjee-syamantak/Recommender_System_with_Deep_Learning-RBM-SAE-",
    "https://github.com/mukherjee-syamantak/ci-cd-final-project",
    "https://github.com/mukherjee-syamantak/Machine_Learning"
  ],
  "certification": [
    "https://www.coursera.org/account/accomplishments/verify/J4MJ6VAHRMHY?utm_source=link&utm_medium=certificate&utm_content=cert_image&utm_campaign=sharing_cta&utm_product=course",
    "https://www.coursera.org/account/accomplishments/verify/KNBH7HXYNQM7?utm_source=link&utm_medium=certificate&utm_content=cert_image&utm_campaign=sharing_cta&utm_product=course",
    "https://www.coursera.org/account/accomplishments/verify/FMRT24WQLQGT?utm_source=link&utm_me

### Step 5: Chunking & FAISS Index Construction

In [6]:
def build_enriched_chunks(structured: Dict[str, Any]) -> List[Dict[str, Any]]:
    chunks = []
    doc_hash = structured["document_hash"]

    # 1. Global Contact & Links Summary Chunk (Ensures 100% recall on URL queries)
    link_lines = [f"{k.replace('_', ' ').title()} link: {url}" for k, urls in structured.get("global_links", {}).items() for url in urls]
    if link_lines:
        chunks.append({
            "chunk_id": f"{doc_hash}_links",
            "text": "Section: Contact and Verified Profile Links\n" + "\n".join(link_lines),
            "metadata": {"section": "Contact/Links", "source": "hyperlink_annotation"}
        })

    # 2. Text Chunks enriched with section context
    for page in structured["pages"]:
        text = page["text"]
        if page["links"]:
            text += "\n\nEmbedded Page Hyperlinks:\n" + "\n".join([f"- {l['label']} ({l['type']}): {l['url']}" for l in page["links"]])

        # Simple sliding window chunker
        max_chars, overlap = 1000, 150
        for i in range(0, len(text), max_chars - overlap):
            chunk_str = text[i:i + max_chars].strip()
            if chunk_str:
                chunks.append({
                    "chunk_id": f"{doc_hash}_p{page['page']}_{i}",
                    "text": f"Page {page['page']} Content:\n{chunk_str}",
                    "metadata": {"page": page["page"], "source": "layout_text"}
                })
    return chunks

chunks = build_enriched_chunks(structured_data)

# Initialize Local Embedding Model & Build FAISS Index
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
embeddings = embedding_model.encode([c["text"] for c in chunks], normalize_embeddings=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim) # Cosine similarity via Inner Product on normalized vectors
index.add(np.array(embeddings).astype("float32"))

with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(chunks, f)
faiss.write_index(index, INDEX_PATH)

print(f"FAISS Index successfully generated with {index.ntotal} 384-dimensional vectors.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS Index successfully generated with 8 384-dimensional vectors.


### Step 6: Query Canonicalization & Semantic Cache Engine
Maps paraphrased questions into unified intents and serves matching answers rapidly.

In [7]:
SYNONYMS = {
    "interviewee": "candidate", "applicant": "candidate", "resume holder": "candidate", "cv owner": "candidate",
    "mail id": "email", "email id": "email", "contact email": "email",
    "mobile number": "phone", "contact number": "phone",
    "technologies": "skills", "tech stack": "skills", "technical skills": "skills",
    "work history": "experience", "employment history": "experience",
    "repo": "github", "repository": "github", "source code": "github", "code link": "github",
    "live demo": "project link", "deployed link": "project link", "hosted link": "project link"
}

CANONICAL_PATTERNS = [
    (r"\b(candidate|person|profile|applicant)\b.*\bname\b|\bname\b.*\b(candidate|person|profile|applicant)\b", "candidate name"),
    (r"\b(email|mail)\b", "email"),
    (r"\b(phone|mobile|contact number)\b", "phone"),
    (r"\blinkedin\b", "linkedin link"),
    (r"\b(github|repo|repository|source code)\b", "github link"),
    (r"\b(project link|demo|deployed)\b", "project link"),
    (r"\b(skills|technologies|tech stack)\b", "skills"),
]

def canonicalize_query(query: str) -> str:
    """Maps paraphrased questions into unified canonical concepts."""
    q = re.sub(r"[^\w\s]", " ", query.lower()).strip()
    q = re.sub(r"\s+", " ", q)
    for phrase, replacement in sorted(SYNONYMS.items(), key=lambda x: len(x[0]), reverse=True):
        q = re.sub(r"\b" + re.escape(phrase) + r"\b", replacement, q)
    q = re.sub(r"\s+", " ", q).strip()

    for pattern, canonical in CANONICAL_PATTERNS:
        if re.search(pattern, q):
            return canonical
    return q

def load_cache() -> List[Dict[str, Any]]:
    if os.path.exists(CACHE_PATH):
        with open(CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    return []

def save_cache(cache: List[Dict[str, Any]]):
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, indent=2, ensure_ascii=False)

def check_semantic_cache(query: str, doc_hash: str) -> Dict[str, Any]:
    cache = load_cache()
    canonical = canonicalize_query(query)

    # Layer 1: Exact Canonical Match (0ms latency, 0 compute)
    for item in cache:
        if item["document_hash"] == doc_hash and item["canonical_query"] == canonical:
            return {"hit": True, "answer": item["answer"], "source": "⚡ EXACT CANONICAL CACHE HIT", "matched_query": item["original_query"], "score": 1.0}

    # Layer 2: Semantic Embedding Similarity
    if cache:
        q_emb = embedding_model.encode([canonical], normalize_embeddings=True)[0]
        best_match, max_score = None, -1.0

        for item in cache:
            if item["document_hash"] != doc_hash:
                continue
            score = float(np.dot(q_emb, np.array(item["embedding"], dtype=np.float32)))
            if score > max_score:
                max_score, best_match = score, item

        if best_match and max_score >= SEMANTIC_CACHE_THRESHOLD:
            return {"hit": True, "answer": best_match["answer"], "source": "🧠 SEMANTIC CACHE HIT", "matched_query": best_match["original_query"], "score": max_score}

    return {"hit": False, "canonical_query": canonical}

def add_to_cache(query: str, canonical: str, answer: str, doc_hash: str):
    cache = load_cache()
    emb = embedding_model.encode([canonical], normalize_embeddings=True)[0].tolist()

    for item in cache:
        if item["document_hash"] == doc_hash and item["canonical_query"] == canonical:
            item.update({"answer": answer, "original_query": query, "embedding": emb})
            save_cache(cache)
            return

    cache.append({"document_hash": doc_hash, "original_query": query, "canonical_query": canonical, "answer": answer, "embedding": emb})
    save_cache(cache)

### Step 7: Retrieval & Local QA Pipeline

In [8]:

from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch

# 1. Initialize tokenizer and model directly
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)

# Move to GPU if available
if DEVICE == 0:
    qa_model = qa_model.to("cuda")

def answer_query(query: str) -> Dict[str, Any]:
    doc_hash = structured_data["document_hash"]

    # Step 1: Check Cache
    cached_result = check_semantic_cache(query, doc_hash)
    if cached_result["hit"]:
        return cached_result

    canonical = cached_result["canonical_query"]

    # Step 2: Check Fast Structured Link Lookup
    link_mapping = {
        "email": "email",
        "linkedin link": "linkedin",
        "github link": "github",
        "certification link": "certification",
        "project link": "project_demo"
    }

    if canonical in link_mapping:
        urls = structured_data["global_links"].get(link_mapping[canonical], [])
        if urls:
            ans = "\n".join(urls)
            add_to_cache(query, canonical, ans, doc_hash)
            return {"hit": False, "answer": ans, "source": "🔍 STRUCTURED LINK LOOKUP", "score": 1.0}

    # Step 3: FAISS Vector Retrieval
    q_emb = embedding_model.encode([canonical], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_emb, TOP_K)

    retrieved_chunks = [chunks[i] for i in idxs[0] if i != -1]

    if not retrieved_chunks:
        return {"hit": False, "answer": "The resume does not contain information to answer this question.", "source": "🤖 RAG RETRIEVAL", "score": 0.0}

    best_answer = "Information not explicitly found in the resume."
    best_confidence = 0.0

    try:
        # Step 4: Advanced Direct Model Inference
        for chunk in retrieved_chunks:
            # Prepend a context hint to help the model process bullet points
            context_text = "Candidate Resume Document:\n" + chunk["text"]
            inputs = qa_tokenizer(query, context_text, return_tensors="pt", truncation=True, max_length=512)

            if DEVICE == 0:
                inputs = {k: v.to("cuda") for k, v in inputs.items()}

            with torch.no_grad():
                outputs = qa_model(**inputs)

            # Extract logits for the first batch item
            start_logits = outputs.start_logits[0]
            end_logits = outputs.end_logits[0]

            # THE FIX: Force extraction by heavily penalizing the [CLS]/<s> "No Answer" token at index 0
            start_logits[0] = -10000
            end_logits[0] = -10000

            # Convert to pseudo-probabilities
            start_probs = torch.softmax(start_logits, dim=-1)
            end_probs = torch.softmax(end_logits, dim=-1)

            best_chunk_score = -1.0
            best_s = 0
            best_e = 0

            # THE FIX: Sliding window search ensures the End Token always comes AFTER the Start Token
            # We restrict the max answer length to 40 tokens to prevent massive blocks of junk text
            for i in range(1, len(start_probs)):
                for j in range(i, min(i + 40, len(end_probs))):
                    score = (start_probs[i] * end_probs[j]).item()
                    if score > best_chunk_score:
                        best_chunk_score = score
                        best_s = i
                        best_e = j

            # If this chunk's best answer is better than previous chunks, save it
            if best_chunk_score > best_confidence and best_chunk_score > 0.001:
                answer_tokens = inputs["input_ids"][0][best_s : best_e + 1]
                extracted = qa_tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()

                # Filter out single-character garbage answers
                if extracted and len(extracted) > 2:
                    best_answer = extracted
                    best_confidence = best_chunk_score

    except Exception as e:
        return {"hit": False, "answer": f"Extraction error: {e}", "source": "ERROR", "score": 0.0}

    # Step 5: Save new result to Semantic Cache
    add_to_cache(query, canonical, best_answer, doc_hash)

    return {
        "hit": False,
        "answer": best_answer,
        "source": "🤖 RAG RETRIEVAL + LOCAL LLM QA",
        "score": float(scores[0][0]) if len(scores[0]) > 0 else 0.0
    }

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/496M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### Step 8: Verification & Testing
Simulates user queries to demonstrate cache hits vs live LLM evaluation.

In [9]:
test_queries = [
    "What is the candidate name?",      # First ask: Will trigger RAG + QA
    "Who is the interviewee name?",     # Second ask (Paraphrased): Should trigger ⚡ EXACT CANONICAL CACHE HIT
    "Can you tell me the applicant name?", # Third ask: Should trigger 🧠 SEMANTIC CACHE HIT
    "Show me the GitHub link",          # Will extract directly from annotated link lookup
    "Do you have a repo code link?",    # Should trigger ⚡ EXACT CANONICAL CACHE HIT for GitHub
    "What technologies and skills does the candidate know?" # RAG extraction
]

for q in test_queries:
    print("=" * 80)
    print(f"QUERY: \"{q}\"")
    res = answer_query(q)
    print(f"STATUS: {res['source']}")
    if "matched_query" in res:
        print(f"MATCHED PREVIOUS: \"{res['matched_query']}\" (Sim Score: {res['score']:.3f})")
    print(f"ANSWER:\n{res['answer']}\n")

QUERY: "What is the candidate name?"
STATUS: 🤖 RAG RETRIEVAL + LOCAL LLM QA
ANSWER:
Syamantak Mukherjee

QUERY: "Who is the interviewee name?"
STATUS: ⚡ EXACT CANONICAL CACHE HIT
MATCHED PREVIOUS: "What is the candidate name?" (Sim Score: 1.000)
ANSWER:
Syamantak Mukherjee

QUERY: "Can you tell me the applicant name?"
STATUS: ⚡ EXACT CANONICAL CACHE HIT
MATCHED PREVIOUS: "What is the candidate name?" (Sim Score: 1.000)
ANSWER:
Syamantak Mukherjee

QUERY: "Show me the GitHub link"
STATUS: 🔍 STRUCTURED LINK LOOKUP
ANSWER:
https://github.com/mukherjee-syamantak
https://github.com/mukherjee-syamantak/LLM-Operations_Workflow
https://github.com/mukherjee-syamantak/Recommender_System_with_Deep_Learning-RBM-SAE-
https://github.com/mukherjee-syamantak/ci-cd-final-project
https://github.com/mukherjee-syamantak/Machine_Learning

QUERY: "Do you have a repo code link?"
STATUS: ⚡ EXACT CANONICAL CACHE HIT
MATCHED PREVIOUS: "Show me the GitHub link" (Sim Score: 1.000)
ANSWER:
https://github.com/mukhe

In [ ]:
# additional code section for debugging only. Not for final execution!
import os
if os.path.exists(CACHE_PATH):
    os.remove(CACHE_PATH)
    print("Old cache deleted. Ready for a fresh run!")